# CN7031 · Week 4 — Apache Hive: SQL on Hadoop

---

**Practical lab workbook | Big Data Analytics (CN7031) 2026-27**
**Dr Fahimeh Jafari** : [F.Jafari@uel.ac.uk](mailto:F.Jafari@uel.ac.uk)

---

### Learning outcomes
By the end of this lab, you should be able to:
1. Run **Apache Hive** on a single-node HDFS cluster in Google Colab.
2. Create **databases, tables and views** in Hive, and load data files into them.
3. Write queries in **HiveQL**, Hive's SQL dialect, including **joins**.
4. Use **analytic (window) functions** such as `RANK`, `DENSE_RANK` and `NTILE`.
5. Speed up queries with **partitioning, bucketing** and columnar storage.
6. (Extra) Count words with **Apache Pig** and its Grunt shell.

### What is Hive?
HDFS stores files, and MapReduce processes them, but writing a MapReduce program in Java for every question is slow work. **Apache Hive** lets you ask the same questions in **SQL**:

- a **table** in Hive is a **folder of files in HDFS**, plus a description of its columns;
- the descriptions live in the **metastore**, a small database of table definitions;
- **HiveServer2** receives your SQL, turns it into **MapReduce jobs**, runs them over the files, and returns the result.

Hive reads the files only when you query them, and applies the column definitions at that moment: this is **schema-on-read**. In Week 3 you supplied a schema yourself to query Sqoop's files; here Hive does it for you.

### Three kinds of cell
| A cell starting with… | Behaves like… | You type… |
| :-- | :-- | :-- |
| `%%bash` (or a line starting with `!`) | a **Linux terminal** | `hdfs dfs …` and other shell commands |
| `%%hive` | the **Hive editor** in Hue (or the `beeline` command line) | HiveQL statements, each ending with `;` |
| `%%pig` | the **Grunt shell** of Apache Pig | Pig Latin statements (Section 6) |

A `%%hive` cell keeps the same Hive session as the previous one: after `USE userdb6;`, every later cell works in `userdb6`, as in the Hue editor. Cells with a **▶ title** and hidden code prepare the lab; run them, but you do not need to read or type their code.

### How to use the notebook
- Use **Google Colab → Python 3 → CPU**. Run the cells in order, starting from Section 1. On a fresh runtime, Section 1 takes roughly 6–8 minutes.
- Download the four data files from Moodle (`Sample.txt`, `Customers.txt`, `Orders.txt`, `wordcount_simple.txt`); you upload them in Section 2.
- Everything runs on **one temporary Colab VM**: one NameNode, one DataNode and one HiveServer2. Hive's MapReduce jobs run in Hadoop's **local mode** rather than on a YARN cluster. This is a teaching demonstration, **not** a production cluster.

**Submitting your work (optional):** take screenshots of your results wherever you like, paste them into a Word file, and submit the Word or PDF file through the submission link on Moodle.

**Important:** Colab storage is temporary and is deleted when its runtime resets. If that happens, run Section 1 again and upload the files again.

## 1. Start HDFS and Hive

**Goal:** Prepare the pieces Hive needs on one Colab VM.

| Step | What it sets up | Role in this lab |
| :-- | :-- | :-- |
| **1.1** | Hadoop 3.5.0 (one NameNode + one DataNode) | HDFS stores every table's files |
| **1.2** | **Apache Hive 4.1.0**: metastore, HiveServer2 and a `beeline` session | Turns HiveQL into MapReduce jobs over those files |
| **1.3** | A quick check that every service is running | Catches set-up problems before the exercises |

Both set-up cells are safe to rerun in the **same runtime**: they skip work that is already done.

### 1.1 Start HDFS

**Goal:** Configure Hadoop 3.5.0 exactly as in Weeks 2 and 3: format the NameNode **once** on fresh storage, and start one NameNode and one DataNode. MapReduce is set to **local mode**, so Hive's jobs run inside HiveServer2 instead of on a YARN cluster.

**Expected output:** `HDFS is ready ✓`, then `NameNode running · 1 DataNode running`. The first run downloads Hadoop and takes a few minutes.

In [1]:
# @title ▶ 1.1 Set up Hadoop 3.5.0 (one NameNode + one DataNode)
# Import paths, operating-system utilities, download and checksum tools.
from pathlib import Path
import os
import subprocess
import urllib.request
import urllib.error
import hashlib
import re
import tarfile
import time

# Import XML tools to create Hadoop and Hive configuration files.
from xml.etree import ElementTree as ET

# Import the HTTP client used to check the NameNode and DataNode web services.
import requests

# Set a persistent path within the current temporary Colab runtime.
LAB = Path('/content/cn7031-week4-hive')
# Create the working directory when it does not exist.
LAB.mkdir(parents=True, exist_ok=True)
# Specify the Hadoop and Java paths.
HADOOP = LAB / 'hadoop-3.5.0'
JAVA = Path('/usr/lib/jvm/java-17-openjdk-amd64')

# Run a system command and stop with a useful error if it fails.
def checked(args, timeout=600, env=None, cwd=None):
    # Capture standard output and errors as readable text.
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout, env=env, cwd=cwd)
    # Check the exit status before using the command's result.
    if p.returncode:
        raise RuntimeError(p.stdout[-5000:])
    # Return the printed output to the caller.
    return p.stdout

# Match a digest's length (in hexadecimal characters) to the algorithm that produced it.
DIGEST_BY_LENGTH = {32: 'md5', 40: 'sha1', 64: 'sha256', 128: 'sha512'}

# Read the first checksum file a publisher provides next to a download.
def published_digest(url, suffixes):
    # Remember the filename so it can be removed from the checksum text.
    name = url.rsplit('/', 1)[-1]
    for suffix in suffixes:
        try:
            # Download the small checksum file, e.g. file.tar.gz.sha512.
            with urllib.request.urlopen(url + suffix, timeout=60) as response:
                text = response.read().decode(errors='replace')
        except (urllib.error.URLError, TimeoutError):
            # Try the next checksum format if this one is not published.
            continue
        # Remove the filename, then keep only the hexadecimal groups.
        tokens = re.split(r'[\s:=()*]+', text.replace(name, ' '))
        digest = ''.join(t for t in tokens if re.fullmatch(r'[0-9a-fA-F]+', t)).lower()
        # Return the algorithm and digest when the length is recognised.
        if len(digest) in DIGEST_BY_LENGTH:
            return DIGEST_BY_LENGTH[len(digest)], digest
    # No usable checksum file was found.
    return None, None

# Download a file, showing progress, and keep it only if it matches the published checksum.
def download_verified(url, dest, suffixes, required=True, checksum_url=None):
    # Download into a temporary name so a failed check leaves nothing behind.
    partial = dest.with_name(dest.name + '.part')
    # timeout=60: a connection that stalls for a minute raises an error instead of hanging forever.
    with urllib.request.urlopen(url, timeout=60) as response, partial.open('wb') as out:
        total = int(response.headers.get('Content-Length') or 0)
        done, step = 0, -1
        while chunk := response.read(1 << 20):
            out.write(chunk)
            done += len(chunk)
            # Report progress in 10% steps for large downloads only.
            if total > 50_000_000 and done * 10 // total > step:
                step = done * 10 // total
                print(f'  Download: {step * 10}%  ({done >> 20} of {total >> 20} MB)')
    # Look up the publisher's checksum for this exact file (a few attempts: the file is tiny but sites can be slow).
    for attempt in range(3):
        algorithm, expected = published_digest(checksum_url or url, suffixes)
        if algorithm:
            break
        time.sleep(3)
    if algorithm is None:
        # Stop, unless this download is allowed to proceed without a checksum.
        if required:
            partial.unlink()
            raise RuntimeError(f'No checksum published for {url}.')
        print(f'  Note: no checksum file found for {dest.name}; continuing without one.')
    else:
        # Calculate the digest of the downloaded file with the same algorithm.
        with partial.open('rb') as stream:
            actual = hashlib.file_digest(stream, algorithm).hexdigest()
        # Reject the file if the digests do not match.
        if actual != expected:
            partial.unlink()
            raise RuntimeError(f'Download checksum failed for {dest.name}.')
        print(f'  {algorithm.upper()} checksum verified: {dest.name}')
    # Give the verified file its final name.
    partial.replace(dest)

# Where to download Apache releases from, fastest first. The CDN and main site carry only the newest
# releases; a public mirror keeps older ones; the Apache archive keeps every release but can be slow.
APACHE_SITES = ('https://dlcdn.apache.org', 'https://downloads.apache.org',
                'https://repo.huaweicloud.com/apache', 'https://archive.apache.org/dist')
APACHE_ARCHIVE = 'https://archive.apache.org/dist'

# Download an Apache release, trying each site in turn (twice each) until one works.
def download_apache(path, dest, suffixes, required=True):
    problems = []
    for site in APACHE_SITES:
        for attempt in (1, 2):
            try:
                print(f'  Trying {site} …')
                # Checksums always come from Apache itself, even when the file comes from a mirror.
                official = site if 'apache.org' in site else APACHE_ARCHIVE
                download_verified(f'{site}/{path}', dest, suffixes, required, f'{official}/{path}')
                return
            except urllib.error.HTTPError as error:
                # Not on this site (e.g. 404 after a release moves to the archive): try the next site.
                problems.append(f'{site}: HTTP {error.code}')
                break
            except (OSError, RuntimeError) as error:
                # Timeout, dropped connection or bad checksum: wait, then retry.
                problems.append(f'{site}: {error}')
                print(f'  Problem: {error}. Retrying …')
                time.sleep(5)
    raise RuntimeError('Could not download ' + path + ' from any Apache site:\n' + '\n'.join(problems))

# Install Java 17 only when it is not already available.
if not (JAVA / 'bin/java').exists():
    # Update package metadata for the Colab VM.
    print('Installing Java …')
    checked(['apt-get', 'update', '-qq'])
    # Install the headless JDK required by Hadoop, Hive and Pig.
    checked(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'])

# Download, verify and unpack Hadoop only once per current VM.
if not (HADOOP / '.verified').exists():
    print('Downloading Hadoop from Apache (about 554 MB) …')
    archive = LAB / 'hadoop-3.5.0.tar.gz'
    download_apache('hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz', archive, ('.sha512',))
    # Extract the verified archive safely.
    print('Installing Hadoop …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    # Record successful installation and remove the compressed archive.
    (HADOOP / '.verified').write_text('ok')
    archive.unlink()

# Set Java and Hadoop environment variables for child processes (!, %%bash, hive, pig).
os.environ.update(JAVA_HOME=str(JAVA), HADOOP_HOME=str(HADOOP),
                  HADOOP_CONF_DIR=str(HADOOP / 'etc/hadoop'),
                  HADOOP_LOG_DIR=str(LAB / 'logs'), HADOOP_PID_DIR=str(LAB / 'pids'),
                  HADOOP_COMMON_HOME=str(HADOOP), HADOOP_MAPRED_HOME=str(HADOOP))
# Open the JDK internals that Hadoop, Hive and Pig still use on Java 17.
OPENS = ' '.join(f'--add-opens={m}=ALL-UNNAMED' for m in (
    'java.base/java.lang', 'java.base/java.lang.invoke', 'java.base/java.lang.reflect',
    'java.base/java.io', 'java.base/java.net', 'java.base/java.nio', 'java.base/java.util',
    'java.base/java.util.concurrent', 'java.base/java.util.concurrent.atomic',
    'java.base/java.util.regex', 'java.base/java.text', 'java.base/sun.nio.ch',
    'java.base/sun.nio.cs', 'java.base/sun.security.action', 'java.base/sun.util.calendar',
    'java.sql/java.sql'))
os.environ['HADOOP_CLIENT_OPTS'] = f'-Xmx1g {OPENS}'
# Put Hadoop and Java executables on the shell PATH.
os.environ['PATH'] = f'{HADOOP}/bin:{JAVA}/bin:' + os.environ['PATH']
# Make separate folders for namespace metadata, blocks, logs and process IDs.
for name in ('namenode', 'datanode', 'logs', 'pids'):
    (LAB / name).mkdir(exist_ok=True)

# Write a Hadoop-style XML configuration file (Hadoop's config folder unless another is given).
def config(name, values, folder=None):
    # Create the configuration root element.
    root = ET.Element('configuration')
    # Add each property name and its value.
    for key, value in values.items():
        property_node = ET.SubElement(root, 'property')
        ET.SubElement(property_node, 'name').text = key
        ET.SubElement(property_node, 'value').text = str(value)
    # Save the resulting configuration file.
    ET.ElementTree(root).write(Path(folder or HADOOP / 'etc/hadoop') / name, encoding='utf-8')

# Point HDFS clients at the local NameNode using port 9000.
config('core-site.xml', {
    'fs.defaultFS': 'hdfs://127.0.0.1:9000',
    'hadoop.tmp.dir': str(LAB / 'tmp'),
})
# Assign local storage directories and a default replication factor of one.
config('hdfs-site.xml', {
    'dfs.namenode.name.dir': (LAB / 'namenode').as_uri(),
    'dfs.datanode.data.dir': (LAB / 'datanode').as_uri(),
    'dfs.replication': 1,
    # Bind RPC and health-report endpoints to the VM's loopback address.
    'dfs.namenode.rpc-address': '127.0.0.1:9000',
    'dfs.namenode.rpc-bind-host': '127.0.0.1',
    'dfs.namenode.http-address': '127.0.0.1:9870',
    'dfs.namenode.http-bind-host': '127.0.0.1',
    # Keep DataNode transfer, HTTP and IPC endpoints on localhost.
    'dfs.datanode.address': '127.0.0.1:9866',
    'dfs.datanode.http.address': '127.0.0.1:9864',
    'dfs.datanode.ipc.address': '127.0.0.1:9867',
    'dfs.datanode.hostname': '127.0.0.1',
    'dfs.client.use.datanode.hostname': 'true',
    'dfs.datanode.use.datanode.hostname': 'true',
})
# Run Hive's MapReduce jobs in local mode (inside the HiveServer2 process, no YARN cluster).
config('mapred-site.xml', {
    'mapreduce.framework.name': 'local',
    # Keep each job's temporary staging files in one HDFS folder.
    'mapreduce.jobtracker.staging.root.dir': '/tmp/hadoop-staging',
})

# Add Java and a default heap size to Hadoop's environment script once.
# The heap can still be raised per process (HiveServer2 gets more in Step 1.2).
env = HADOOP / 'etc/hadoop/hadoop-env.sh'
if 'CN7031 environment' not in env.read_text():
    env.write_text(env.read_text() + f'\n# CN7031 environment\nexport JAVA_HOME={JAVA}\n'
                   'export HADOOP_HEAPSIZE_MAX=${HADOOP_HEAPSIZE_MAX:-512}\n')

# Show only warnings and errors from Hadoop's internals, so commands print short output.
log4j = HADOOP / 'etc/hadoop/log4j.properties'
QUIET = ['log4j.logger.org.apache.hadoop=WARN',
         'log4j.logger.org.apache.hadoop.fs.FileUtil=ERROR',
         'log4j.logger.org.apache.hadoop.mapred.LocalDistributedCacheManager=ERROR',
         'log4j.logger.org.apache.hadoop.metrics2.impl.MetricsSystemImpl=ERROR']
if log4j.exists():
    text = log4j.read_text()
    missing = [line for line in QUIET if line not in text]
    if missing:
        log4j.write_text(text + '\n# CN7031: quieter client output\n' + '\n'.join(missing) + '\n')

# Format only when no NameNode filesystem already exists.
if not (LAB / 'namenode/current/VERSION').exists():
    # Never silently overwrite an unexpectedly populated storage directory.
    assert all(not any((LAB / name).iterdir()) for name in ('namenode', 'datanode')), \
        'Existing storage found. Do not reformat it.'
    # Give the new HDFS namespace a fixed educational cluster ID.
    checked(['hdfs', 'namenode', '-format', '-nonInteractive', '-clusterid', 'CID-cn7031-week4'])

# Start a service only if its local HTTP endpoint is not responding.
for daemon, port in [('namenode', 9870), ('datanode', 9864)]:
    try:
        # Raise an error on a missing or unhealthy service endpoint.
        requests.get(f'http://127.0.0.1:{port}/jmx', timeout=2).raise_for_status()
    except requests.RequestException:
        # Start the selected Hadoop daemon in the background.
        checked(['hdfs', '--daemon', 'start', daemon])

# Wait until the NameNode reports exactly one live DataNode.
for attempt in range(45):
    try:
        # Query NameNode's JMX metrics over its local HTTP endpoint.
        response = requests.get('http://127.0.0.1:9870/jmx',
                                params={'qry': 'Hadoop:service=NameNode,name=FSNamesystemState'},
                                timeout=3)
        response.raise_for_status()
        # Exit the retry loop when the teaching cluster is ready.
        if response.json()['beans'][0]['NumLiveDataNodes'] == 1:
            break
    except (requests.RequestException, KeyError, IndexError):
        # Retry while the services finish their startup sequence.
        pass
    # Pause two seconds before asking again.
    time.sleep(2)
else:
    # Stop if the DataNode never becomes available.
    raise RuntimeError(f'Startup failed. Inspect the Hadoop logs in {LAB / "logs"}.')

# Wait until the NameNode has left safe mode, allowing writes.
checked(['hdfs', 'dfsadmin', '-safemode', 'wait'], timeout=90)
# Confirm that HDFS is usable before Hive is installed.
print('\nHDFS is ready ✓')
print('Hadoop 3.5.0 · NameNode running · 1 DataNode running')

Installing Java …
  Trying https://dlcdn.apache.org …
  Download: 0%  (1 of 554 MB)
  Download: 10%  (56 of 554 MB)
  Download: 20%  (111 of 554 MB)
  Download: 30%  (167 of 554 MB)
  Download: 40%  (222 of 554 MB)
  Download: 50%  (278 of 554 MB)
  Download: 60%  (333 of 554 MB)
  Download: 70%  (389 of 554 MB)
  Download: 80%  (444 of 554 MB)
  Download: 90%  (499 of 554 MB)
  Download: 100%  (554 of 554 MB)
  SHA512 checksum verified: hadoop-3.5.0.tar.gz
Installing Hadoop …

HDFS is ready ✓
Hadoop 3.5.0 · NameNode running · 1 DataNode running


### 1.2 Install Hive and start HiveServer2

**Goal:** Install Apache Hive 4.1.0 and start its three parts:

| Part | Job | In this lab |
| :-- | :-- | :-- |
| **Metastore** | Remembers every database, table, column and file location | A small Derby database on Colab's disk |
| **HiveServer2** | Accepts SQL from clients, compiles it into MapReduce jobs and runs them | A Java service on port 10000 |
| **beeline** | Hive's command-line client; Hue's Hive editor talks to the same server | One session, used by every `%%hive` cell |

Table **data** is not stored in the metastore: each table is a folder in HDFS under `/user/hive/warehouse`.

**Expected output:** the `Connected to: Apache Hive (version 4.1.0)` line, then `Hive is ready ✓`. The first run downloads Hive and takes a few minutes.

In [2]:
# @title ▶ 1.2 Install Apache Hive 4.1.0, start HiveServer2 and turn on %%hive cells
# Import process, network, text and notebook tools.
import os
import re
import shutil
import socket
import subprocess
import tarfile
import time
from pathlib import Path

import pexpect
from IPython.core.magic import register_cell_magic

# This cell needs the HDFS set-up from 1.1.
if 'LAB' not in globals() or 'download_apache' not in globals():
    raise RuntimeError('Run cell 1.1 first (in this runtime), then run this cell again.')

# Hive 4.1 is the release line that runs on Java 17, like this Hadoop.
HIVE_VERSION = '4.1.0'
HIVE = LAB / f'apache-hive-{HIVE_VERSION}-bin'
HIVE_CONF = HIVE / 'conf'

# Download, verify and unpack Hive only once per current VM.
if not (HIVE / '.verified').exists():
    print(f'Downloading Apache Hive {HIVE_VERSION} (about 470 MB) …')
    archive = LAB / 'hive.tar.gz'
    download_apache(f'hive/hive-{HIVE_VERSION}/apache-hive-{HIVE_VERSION}-bin.tar.gz', archive,
                    ('.sha256', '.sha512'))
    print('Installing Hive …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    for script in (HIVE / 'bin').iterdir():
        script.chmod(script.stat().st_mode | 0o111)
    (HIVE / '.verified').write_text('ok')
    archive.unlink()

# Hive and Hadoop each ship Google Guava; keep only Hadoop's newer copy so both use the same version.
hadoop_guava = sorted((HADOOP / 'share/hadoop/common/lib').glob('guava-*.jar'))
for old in (HIVE / 'lib').glob('guava-*.jar'):
    if hadoop_guava and old.name != hadoop_guava[-1].name:
        old.unlink()
        shutil.copy(hadoop_guava[-1], HIVE / 'lib')

# Folders Hive keeps on the local disk: metastore database, scratch space and logs.
for name in ('hive-local', 'hive-resources', 'hive-querylog', 'hive-oplogs'):
    (LAB / name).mkdir(exist_ok=True)

# Hive's configuration: an embedded Derby metastore, MapReduce in local mode, and HiveServer2 on port 10000.
config('hive-site.xml', {
    # The metastore: a small Derby database on the local disk that holds every table definition.
    'javax.jdo.option.ConnectionURL': f'jdbc:derby:;databaseName={LAB}/metastore_db;create=true',
    # Table data lives in HDFS under this folder.
    'hive.metastore.warehouse.dir': '/user/hive/warehouse',
    # Keep plain managed tables as they are (Hive 4 would otherwise translate them into external
    # tables stored elsewhere), so every table lives under /user/hive/warehouse/<database>.db/.
    'metastore.metadata.transformer.class': ' ',
    'metastore.warehouse.external.dir': '/user/hive/warehouse',
    'hive.metastore.warehouse.external.dir': '/user/hive/warehouse',
    # Run queries as MapReduce jobs (local mode, see mapred-site.xml).
    'hive.execution.engine': 'mr',
    'hive.exec.submitviachild': 'false',
    'hive.exec.submit.local.task.via.child': 'false',
    # Scratch and log folders.
    'hive.exec.scratchdir': '/tmp/hive',
    'hive.exec.local.scratchdir': str(LAB / 'hive-local'),
    'hive.downloaded.resources.dir': str(LAB / 'hive-resources'),
    'hive.querylog.location': str(LAB / 'hive-querylog'),
    'hive.server2.logging.operation.log.location': str(LAB / 'hive-oplogs'),
    # HiveServer2: listen on this machine only, and run queries as the connected user's process.
    'hive.server2.thrift.bind.host': '127.0.0.1',
    'hive.server2.thrift.port': 10000,
    'hive.server2.enable.doAs': 'false',
    # Do not send Hive's INFO/WARN progress log to beeline, so each cell shows only its result.
    'hive.server2.logging.operation.enabled': 'false',
    # Show column names without the table prefix (eid rather than employee.eid).
    'hive.resultset.use.unique.column.names': 'false',
    # Skip column statistics after every INSERT, which only slows a teaching cluster down.
    'hive.stats.column.autogather': 'false',
    'hive.metastore.event.db.notification.api.auth': 'false',
}, folder=HIVE_CONF)

# Environment for Hive's own programs (schematool, HiveServer2, beeline).
HIVE_ENV = {**os.environ, 'HIVE_HOME': str(HIVE), 'HIVE_CONF_DIR': str(HIVE_CONF), 'TERM': 'dumb',
            'HADOOP_HEAPSIZE_MAX': '2048', 'HADOOP_CLIENT_OPTS': f'-Xmx2g {OPENS}'}
os.environ.update(HIVE_HOME=str(HIVE), HIVE_CONF_DIR=str(HIVE_CONF))
os.environ['PATH'] = f'{HIVE}/bin:' + os.environ['PATH']

# HDFS folders Hive needs: scratch space and the warehouse.
checked(['hdfs', 'dfs', '-mkdir', '-p', '/tmp/hive', '/user/hive/warehouse'])
checked(['hdfs', 'dfs', '-chmod', '-R', '777', '/tmp'])

# Create the metastore database once. If an earlier attempt stopped half-way, start it again cleanly.
METASTORE_READY = LAB / '.metastore_ready'
if not METASTORE_READY.exists():
    print('Creating the Hive metastore …')
    shutil.rmtree(LAB / 'metastore_db', ignore_errors=True)
    checked([str(HIVE / 'bin/schematool'), '-dbType', 'derby', '-initSchema'],
            timeout=900, env=HIVE_ENV, cwd=LAB)
    METASTORE_READY.write_text('ok')

# Is HiveServer2 accepting connections on port 10000?
def hiveserver2_up():
    try:
        socket.create_connection(('127.0.0.1', 10000), timeout=2).close()
        return True
    except OSError:
        return False

# Start HiveServer2 in the background once, and wait until it accepts connections.
if not hiveserver2_up():
    print('Starting HiveServer2 (1–3 minutes) …')
    (LAB / 'logs').mkdir(exist_ok=True)
    log = open(LAB / 'logs/hiveserver2.out', 'ab')
    server = subprocess.Popen([str(HIVE / 'bin/hive'), '--service', 'hiveserver2'], stdout=log,
                              stderr=subprocess.STDOUT, env=HIVE_ENV, cwd=LAB, start_new_session=True)
    # Wait up to 5 minutes, and stop early if the server process exits with an error.
    for attempt in range(150):
        if hiveserver2_up() or server.poll() is not None:
            break
        time.sleep(2)
    if not hiveserver2_up():
        tail = (LAB / 'logs/hiveserver2.out').read_text(errors='replace').splitlines()[-40:]
        raise RuntimeError('HiveServer2 did not start. Last log lines:\n' + '\n'.join(tail))

# A long-running command-line shell (beeline or Pig's grunt) that notebook cells type into.
# State such as USE database or SET options is kept between cells, as in a terminal.
class ShellSession:
    def __init__(self, argv, prompt, run_command, echo_command, env, cwd):
        self.argv, self.prompt = argv, prompt
        self.run_command, self.echo_command = run_command, echo_command
        self.env, self.cwd, self.child, self.cells = env, cwd, None, 0

    # Start the shell and wait for its first prompt.
    def start(self, timeout=300):
        self.child = pexpect.spawn(self.argv[0], self.argv[1:], env=self.env, cwd=str(self.cwd),
                                   encoding='utf-8', codec_errors='replace', timeout=1800,
                                   dimensions=(1000, 1000))
        self.child.setecho(False)
        self.child.expect(self.prompt, timeout=timeout)
        return self.tidy(self.child.before)

    # Remove carriage returns, colour codes, bare prompts and the commands this class typed itself.
    def tidy(self, text, typed=()):
        text = re.sub(r'\x1b\[[0-9;?]*[A-Za-z]', '', text.replace('\r', ''))
        lines = []
        for line in text.split('\n'):
            if '__CN7031_DONE_' in line or any(t in line for t in typed):
                continue
            # Drop prompts at the start of a line (the shell may print several in a row).
            bare = re.sub(rf'^\s*(?:(?:{self.prompt})\s*)+', '', line)
            if bare != line and not bare.strip():
                continue
            lines.append(bare)
        return '\n'.join(lines).strip('\n')

    # Send one cell: save it as a script, run it in the shell, and wait for a unique end marker.
    def run(self, text):
        if self.child is None or not self.child.isalive():
            if self.child is not None:
                print('(The previous session had stopped; a new one was started. '
                      'Run your USE and SET statements again.)')
            self.start()
        self.cells += 1
        script = Path(self.cwd) / f'cell_{self.cells}.txt'
        script.write_text(text + '\n')
        marker = f'__CN7031_DONE_{self.cells}__'
        typed = (self.run_command.format(script), self.echo_command.format(marker))
        for command in typed:
            self.child.sendline(command)
        # The marker is printed by the shell's echo command (not the echoed command line itself).
        # If the cell closed the shell (quit), stop waiting and say how to open a new one.
        if self.child.expect([rf'(?<!echo ){marker}', pexpect.EOF]) == 1:
            output = self.tidy(self.child.before, typed)
            self.child.close()
            return (output + '\n' if output else '') + '(The shell has closed. The next cell will start a new one.)'
        output = self.child.before
        self.child.expect(self.prompt)
        return self.tidy(output, typed)

# Remove full-line -- comments and make sure the last statement ends with a semicolon.
def hiveql(cell):
    lines = [line for line in cell.strip().split('\n') if not line.strip().startswith(('--', '#'))]
    text = '\n'.join(lines).strip()
    return text if text.endswith(';') else text + ';'

# One beeline session connected to HiveServer2, kept open for the whole notebook.
HIVE_SESSION = ShellSession(
    [str(HIVE / 'bin/beeline'), '-u', 'jdbc:hive2://127.0.0.1:10000/default', '-n', 'root', '-p', '',
     '--color=false', '--showWarnings=false', '--outputformat=table'],
    prompt=r'\d+: jdbc:hive2://\S*> ', run_command='!run {}', echo_command='!sh echo {}',
    env=HIVE_ENV, cwd=LAB / 'hive-local')

# Notebook shortcut: a %%hive cell runs its HiveQL in the beeline session, like the Hue Hive editor.
# Keep only what matters in beeline's output: the result table, messages and errors, and the row count
# with its time. Hive's log lines and beeline's echo of the statement (already visible in the cell) are dropped.
def clean_hive_output(output, statement):
    typed = {l.strip().rstrip(';').strip() for l in statement.split('\n') if l.strip()}
    kept, in_echo = [], False
    for line in output.split('\n'):
        if line.startswith('>>>  '):
            in_echo = not line.rstrip().endswith(';')
            continue
        if in_echo:
            in_echo = not line.rstrip().endswith(';')
            continue
        if re.match(r'(INFO|WARN|DEBUG)\s+:', line) or line.strip().rstrip(';').strip() in typed:
            continue
        kept.append(line)
    return '\n'.join(kept).strip('\n')

@register_cell_magic
def hive(line, cell):
    text = hiveql(cell)
    if not text.strip(' ;\n'):
        print('(Nothing to run yet: type a HiveQL statement in this cell, ending with ;)')
        return
    print(clean_hive_output(HIVE_SESSION.run(text), text))

# Allow a comment line above %%hive or %%pig: move the %% line to the top, as Jupyter requires.
def comment_before_cell_magic(lines):
    first = next((k for k, l in enumerate(lines) if l.strip() and not l.lstrip().startswith('#')), None)
    if first and lines[first].startswith('%%'):
        return [lines[first]] + lines[:first] + lines[first + 1:]
    return lines

_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [f for f in _ip.input_transformers_cleanup
                                     if getattr(f, '__name__', '') != 'comment_before_cell_magic']
_ip.input_transformers_cleanup.append(comment_before_cell_magic)

# Connect beeline now, so the first %%hive cell answers quickly.
if HIVE_SESSION.child is None or not HIVE_SESSION.child.isalive():
    banner = HIVE_SESSION.start()
    print('\n'.join(l for l in banner.split('\n') if l.startswith(('Connected to', 'Driver'))))
print('\nHive is ready ✓  HiveServer2 on port 10000 · %%hive cells are ready ✓')

  Trying https://dlcdn.apache.org …
  Trying https://downloads.apache.org …
  Trying https://repo.huaweicloud.com/apache …
  Download: 0%  (1 of 469 MB)
  Download: 10%  (47 of 469 MB)
  Download: 20%  (94 of 469 MB)
  Download: 30%  (141 of 469 MB)
  Download: 40%  (188 of 469 MB)
  Download: 50%  (235 of 469 MB)
  Download: 60%  (282 of 469 MB)
  Download: 70%  (329 of 469 MB)
  Download: 80%  (376 of 469 MB)
  Download: 90%  (423 of 469 MB)
  Download: 100%  (469 of 469 MB)
  SHA256 checksum verified: hive.tar.gz
Installing Hive …
Creating the Hive metastore …
Starting HiveServer2 (1–3 minutes) …
Connected to: Apache Hive (version 4.1.0)
Driver: Hive JDBC (version 4.1.0)

Hive is ready ✓  HiveServer2 on port 10000 · %%hive cells are ready ✓


### 1.3 Check the running services

**Goal:** Confirm that every service started.

`jps` lists the Java processes on this machine. You should see **NameNode** and **DataNode** (HDFS), and two **RunJar** processes: HiveServer2, and the beeline client that `%%hive` cells type into. `Jps` is the listing tool itself.

In [3]:
# List the running Java processes: NameNode, DataNode and two RunJar.
!jps

2896 DataNode
4256 Jps
2834 NameNode
4120 RunJar
3773 RunJar


Now send Hive its first statement. `version()` is a built-in Hive function.

Each `%%hive` cell shows only the result: the result table (or a message such as `No rows affected`), then the number of rows and the time Hive took, for example `1 row selected (0.18 seconds)`. Hive's own log messages are hidden to keep the output clean.

In [5]:
# Ask HiveServer2 which version of Hive it is running.
%%hive
SELECT version();


+--------------------------------------------------+
|                       _c0                        |
+--------------------------------------------------+
| 4.1.0 r75e40b7537c91a70ccaa31c397d21823c7528eeb  |
+--------------------------------------------------+
1 row selected (8.342 seconds)


## 2. Upload the data files

**Goal:** Put the four Moodle files on Colab's disk, where Hive and Pig can read them.

| File | Contents | Used in |
| :-- | :-- | :-- |
| `Sample.txt` | 5 employees: id, name, salary, designation, department | Sections 3–5 |
| `Customers.txt` | 7 customers: id, name, age, address, salary | Section 4.3 |
| `Orders.txt` | 4 orders: order id, date, customer id, amount | Section 4.3 |
| `wordcount_simple.txt` | A few sentences about Pig | Section 6 |

In Colab, open the **Files** pane (the folder icon on the left) and drag the four files into it, or use its **Upload** button. They are saved in `/content`.

In [8]:
# List the uploaded text files with their sizes.
!ls -l /content/Customers.txt
!ls -l /content/Orders.txt
!ls -l /content/Sample.txt
!ls -l/content/wordcount_simple.txt

-rw-r--r-- 1 root root 192 Oct 23 15:18 /content/Customers.txt
-rw-r--r-- 1 root root 126 Oct 23 15:18 /content/Orders.txt
-rw-r--r-- 1 root root 164 Oct 23 15:18 /content/Sample.txt
ls: invalid option -- '/'
Try 'ls --help' for more information.


### 2.1 Check the line endings

**Goal:** Make sure Hive will read each line correctly.

Each line of these files holds one record, with the fields separated by a **tab**. A file saved on Windows ends every line with two invisible characters, **carriage return + line feed** (`\r\n`); depending on how a file was downloaded, your copies may or may not have them. Linux and Hive end lines with a line feed alone (`\n`). Hive would keep the `\r` as part of the last column, so the department `Admin` would be stored as `Admin\r`, and `WHERE dept = 'Admin'` would miss it.

`file` reports the line endings: a Windows file is reported as `ASCII text, with CRLF line terminators`, a Linux file as plain `ASCII text`. `cat -A` shows invisible characters: a tab as `^I`, a carriage return as `^M`, and the end of a line as `$`.

In [9]:
# Report the type and line endings of each data file.
!file /content/Customers.txt
!file /content/Orders.txt
!file /content/Sample.txt
!file /content/wordcount_simple.txt

/content/Customers.txt: ASCII text, with CRLF line terminators
/content/Orders.txt: ASCII text, with CRLF line terminators
/content/Sample.txt: ASCII text, with CRLF line terminators
/content/wordcount_simple.txt: ASCII text, with no line terminators


In [10]:
# Show the first two lines of Sample.txt with tabs (^I) and carriage returns (^M) made visible.
!head -n 2 /content/Sample.txt | cat -A

1201^IGopal^I45000^ITechnical^Imanager^M$
1202^IManisha^I45000^IProof^Ireader^M$


`sed -i 's/\r$//'` removes the carriage return at the end of every line, in place (`-i`). It is safe to run even when the files are already fine: it then changes nothing. Then check again: `file` should no longer mention CRLF, and `cat -A` should show lines ending in `$` without `^M`.

In [11]:
# Remove the Windows carriage returns from the three table files.
!sed -i 's/\r$//' /content/Customers.txt /content/Orders.txt /content/Sample.txt

In [12]:
# Check the line endings again: CRLF should be gone.
!file /content/*.txt
!head -n 2 /content/Sample.txt | cat -A

/content/Customers.txt:        ASCII text
/content/Orders.txt:           ASCII text
/content/Sample.txt:           ASCII text
/content/wordcount_simple.txt: ASCII text, with no line terminators
1201^IGopal^I45000^ITechnical^Imanager$
1202^IManisha^I45000^IProof^Ireader$


In [13]:
# Show the first two lines again: they should end in $ without ^M.
%%hive
CREATE SCHEMA userdb6;

No rows affected (0.892 seconds)


## 3. Create a database and a table

**Goal:** Create a Hive database and an `employee` table, load `Sample.txt` into it, and see where Hive keeps the data.

### 3.1 Create a database

**Goal:** Create a database named `userdb` followed by your lab group number. The examples use group 6, so `userdb6`; if you use another number, change it in every cell. `CREATE SCHEMA` and `CREATE DATABASE` mean the same in Hive.

**Expected:** `default` and `userdb6` in the list of databases.

In [15]:
# Create a database for your lab group (here group 6).
%%hive
SHOW DATABASES;

+----------------+
| database_name  |
+----------------+
| default        |
| userdb6        |
+----------------+
2 rows selected (0.338 seconds)


In [18]:
# List the databases.
%%hive
default
userdb6

Error: Error while compiling statement: FAILED: ParseException line 1:0 cannot recognize input near 'default' 'userdb6' '<EOF>'; Query ID: root_20261023152833_80313c13-3c44-4487-bb0b-b9f038c9b728 (state=42000,code=40000)
Aborting command set because "force" is false and command failed: "default
userdb6;"


### 3.2 Select the database

**Goal:** Make `userdb6` the **current database**, so the next statements work inside it. Hive replies `No rows affected`: `USE` changes your session, not any data. Every later `%%hive` cell now works in `userdb6`.

In [19]:
# Make userdb6 the current database for the rest of the session.
%%hive
USE userdb6;


No rows affected (0.041 seconds)


### 3.3 Create the employee table

**Goal:** Describe the table: its columns and their types, and how its data file is laid out.

| Clause | Meaning |
| :-- | :-- |
| `(eid int, name String, …)` | The columns and their types |
| `COMMENT 'Employee details'` | A description stored in the metastore |
| `ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t'` | Columns in the file are separated by tabs |
| `LINES TERMINATED BY '\n'` | Each line of the file is one row |
| `STORED AS TEXTFILE` | The data is kept as plain text files |

The table has **four** columns, but `Sample.txt` has five; you will add the fifth in Step 3.6.

In [20]:
# Create an empty employee table for tab-separated text files.
%%hive
CREATE TABLE IF NOT EXISTS employee (
    eid INT,
    name STRING,
    salary FLOAT,
    designation STRING
)
COMMENT 'Employee details'
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.794 seconds)


### 3.4 Load the data

**Goal:** Put `Sample.txt` into the table. In ordinary SQL you would `INSERT` rows; in Hive you usually **load whole files**:

| Part | Meaning |
| :-- | :-- |
| `LOAD DATA LOCAL INPATH '…'` | Copy a file from the **local** disk of the Hive server (without `LOCAL`, Hive would move a file that is already in HDFS) |
| `OVERWRITE INTO TABLE employee` | Replace any data the table already holds |

In [21]:
# Copy Sample.txt from Colab's disk into the employee table.
%%hive
LOAD DATA LOCAL INPATH '/content/Sample.txt'
OVERWRITE INTO TABLE employee;

No rows affected (2.31 seconds)


In [22]:
# Show the rows of the employee table.
%%hive
SELECT * FROM employee LIMIT 10;

+-------+--------------+----------+--------------+
|  eid  |     name     |  salary  | designation  |
+-------+--------------+----------+--------------+
| 1201  | Gopal        | 45000.0  | Technical    |
| 1202  | Manisha      | 45000.0  | Proof        |
| 1203  | Masthanvali  | 40000.0  | Technical    |
| 1204  | Kiran        | 40000.0  | Hr           |
| 1205  | Kranthi      | 30000.0  | Op           |
+-------+--------------+----------+--------------+
5 rows selected (0.75 seconds)


**Expected:** five employees, from 1201 Gopal to 1205 Kranthi, with salaries shown as `45000.0` because the column is a `Float`.

### 3.5 Where is the data?

**Goal:** See that a Hive table is just a folder of files in HDFS.

`DESCRIBE FORMATTED` prints everything the metastore knows about the table; find the **Location** line. The `hdfs dfs` commands then look inside that folder: `LOAD DATA` simply copied `Sample.txt` there, unchanged.

In [23]:
# Show the full table definition; find the Location line.
%%hive
DESCRIBE FORMATTED employee;

+-------------------------------+----------------------------------------------------+-----------------------+
|           col_name            |                     data_type                      |        comment        |
+-------------------------------+----------------------------------------------------+-----------------------+
| eid                           | int                                                |                       |
| name                          | string                                             |                       |
| salary                        | float                                              |                       |
| designation                   | string                                             |                       |
|                               | NULL                                               | NULL                  |
| # Detailed Table Information  | NULL                                               | NULL                  |
|

In [24]:
# List the files inside the employee table's HDFS folder.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee

Found 1 items
-rw-r--r--   1 root supergroup        160 2026-10-23 15:29 /user/hive/warehouse/userdb6.db/employee/Sample.txt


In [25]:
# Print the table's data file: it is exactly the file you uploaded.
!hdfs dfs -cat /user/hive/warehouse/userdb6.db/employee/Sample.txt

1201	Gopal	45000	Technical	manager
1202	Manisha	45000	Proof	reader
1203	Masthanvali	40000	Technical	writer
1204	Kiran	40000	Hr	Admin
1205	Kranthi	30000	Op	Admin

### 3.6 Add the missing column

**Goal:** Add the fifth column, the department. `ALTER TABLE … ADD COLUMNS` only changes the **definition** in the metastore; the data file stays as it is.

Because Hive applies the definition when it reads the file (**schema-on-read**), the fifth field of every line appears straight away. In a traditional database, adding a column would leave it empty for existing rows.

In [26]:
# Add a dept column to the table definition.
%%hive
ALTER TABLE employee
ADD COLUMNS (dept STRING COMMENT 'Department name');

No rows affected (0.408 seconds)


In [27]:
# Show the table again: the dept column is filled from the same file.
%%hive
SELECT * FROM employee;

+-------+--------------+----------+--------------+----------+
|  eid  |     name     |  salary  | designation  |   dept   |
+-------+--------------+----------+--------------+----------+
| 1201  | Gopal        | 45000.0  | Technical    | manager  |
| 1202  | Manisha      | 45000.0  | Proof        | reader   |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer   |
| 1204  | Kiran        | 40000.0  | Hr           | Admin    |
| 1205  | Kranthi      | 30000.0  | Op           | Admin    |
+-------+--------------+----------+--------------+----------+
5 rows selected (0.451 seconds)


## 4. Work with Hive queries

**Goal:** Practise HiveQL. Most of it is standard SQL; the difference is underneath, where Hive turns each query into MapReduce work over HDFS files.

Simple queries such as `SELECT *` with `WHERE` or `LIMIT` are answered by reading the file directly. Queries that sort, group or join need a **MapReduce job**: these take a few seconds longer, because Hive has to launch and run the job (compare the times in the last line of each output).

### 4.1 Retrieve information

**Goal:** Filter, sort and group the employees.

| Query | Expected |
| :-- | :-- |
| `WHERE salary > 40000` | Gopal and Manisha (45000) |
| `ORDER BY designation` | Hr, Op, Proof, Technical, Technical |
| `GROUP BY salary` | 30000 → 1, 40000 → 2, 45000 → 2 |

In [28]:
# Show the employees who earn more than 40000.
%%hive
SELECT *
FROM employee
WHERE salary > 40000;


+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (1.41 seconds)


In [29]:
# List the employees sorted by designation.
%%hive
SELECT eid, name, dept, designation
FROM employee
ORDER BY designation;


+-------+--------------+----------+--------------+
|  eid  |     name     |   dept   | designation  |
+-------+--------------+----------+--------------+
| 1204  | Kiran        | Admin    | Hr           |
| 1205  | Kranthi      | Admin    | Op           |
| 1202  | Manisha      | reader   | Proof        |
| 1203  | Masthanvali  | writer   | Technical    |
| 1201  | Gopal        | manager  | Technical    |
+-------+--------------+----------+--------------+
5 rows selected (4.895 seconds)


In [30]:
# Count the employees at each salary.
%%hive
SELECT salary, COUNT(*) AS counter
FROM employee
GROUP BY salary;


+----------+----------+
|  salary  | counter  |
+----------+----------+
| 30000.0  | 1        |
| 40000.0  | 2        |
| 45000.0  | 2        |
+----------+----------+
3 rows selected (2.9 seconds)


### 4.2 Create a view

**Goal:** Save a query under a name. A **view** is a stored `SELECT` that you can query like a table: a "virtual table" with no data files of its own. Each time you use it, Hive runs the stored query.

In [31]:
# Save the high-salary query as a view called emp_40000.
%%hive
CREATE VIEW emp_40000 AS
SELECT *
FROM employee
WHERE salary > 40000;


No rows affected (0.9 seconds)


In [32]:
# List the tables and views: emp_40000 is now there.
%%hive
SHOW TABLES;


+------------+
|  tab_name  |
+------------+
| emp_40000  |
| employee   |
+------------+
2 rows selected (0.141 seconds)


In [33]:
# Query the view like a table.
%%hive
SELECT *
FROM emp_40000
LIMIT 5;


+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (0.426 seconds)


In [34]:
# Remove the view (the employee table is not affected).
%%hive
DROP VIEW emp_40000;


No rows affected (0.785 seconds)


### 4.3 Join queries

**Goal:** Combine two tables. Create a `Customers` table and an `Orders` table, load the two files, and join them on the customer id.

Two details differ from the employee table:
- `Customers.txt` stores salaries as `2000.00`, so the `salary` column is a `Float`. As an `int`, Hive could not read `2000.00` and would show `NULL`.
- `Date` is a **reserved word** in HiveQL (it names a data type), so as a column name it must be written in backticks: `` `Date` ``.

In [35]:
# Create the Customers table.
%%hive
CREATE TABLE IF NOT EXISTS Customers (
    ID INT,
    Name STRING,
    Age INT,
    Address STRING,
    Salary FLOAT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;


No rows affected (0.1 seconds)


In [36]:
# Load Customers.txt into the Customers table.
%%hive
LOAD DATA LOCAL INPATH '/content/Customers.txt'
OVERWRITE INTO TABLE Customers;


No rows affected (0.863 seconds)


In [37]:
# Show the customers.
%%hive
SELECT * FROM Customers LIMIT 10;


+-----+-----------+------+------------+----------+
| id  |   name    | age  |  address   |  salary  |
+-----+-----------+------+------------+----------+
| 1   | Ramesh    | 32   | Ahmedabad  | 2000.0   |
| 2   | Khilan    | 25   | Delhi      | 1500.0   |
| 3   | kaushik   | 23   | Kota       | 2000.0   |
| 4   | Chaitali  | 25   | Mumbai     | 6500.0   |
| 5   | Hardik    | 27   | Bhopal     | 8500.0   |
| 6   | Komal     | 22   | MP         | 4500.0   |
| 7   | Muffy     | 24   | Indore     | 10000.0  |
+-----+-----------+------+------------+----------+
7 rows selected (0.365 seconds)


In [38]:
# Create the Orders table; `Date` needs backticks because DATE is a reserved word.
%%hive
CREATE TABLE IF NOT EXISTS Orders (
    OID INT,
    `Date` STRING,
    Customer_ID INT,
    Amount INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;


No rows affected (0.134 seconds)


In [39]:
# Load Orders.txt into the Orders table.
%%hive
LOAD DATA LOCAL INPATH '/content/Orders.txt'
 OVERWRITE INTO TABLE Orders;

No rows affected (0.35 seconds)


In [40]:
# Show the orders.
%%hive
SELECT * FROM Orders;


+------+----------------------+--------------+---------+
| oid  |         date         | customer_id  | amount  |
+------+----------------------+--------------+---------+
| 102  | 2009-10-08 00:00:00  | 3            | 3000    |
| 100  | 2009-10-08 00:00:00  | 3            | 1500    |
| 101  | 2009-11-20 00:00:00  | 2            | 1560    |
| 103  | 2008-05-20 00:00:00  | 4            | 2060    |
+------+----------------------+--------------+---------+
4 rows selected (0.229 seconds)


Now join the tables. `c` and `o` are short **aliases** for the two tables, and `ON (c.ID = o.Customer_ID)` says which rows belong together.

| Join | Keeps | Expected rows |
| :-- | :-- | :-- |
| `JOIN` (inner) | Only customers **with** orders | 4: Khilan, kaushik (twice), Chaitali |
| `LEFT OUTER JOIN` | **Every customer**, with `NULL` where there is no order | 8: the 4 above plus Ramesh, Hardik, Komal, Muffy |
| `RIGHT OUTER JOIN` | **Every order**, with `NULL` where there is no customer | 4: every order has a customer |
| `FULL OUTER JOIN` | Every row of **both** tables | 8 |

In [41]:
# Inner join: customers together with their orders.
%%hive
SELECT c.ID, c.Name, c.Age, o.Amount
FROM Customers c
JOIN Orders o
ON c.ID = o.Customer_ID;


+-----+-----------+------+---------+
| id  |   name    | age  | amount  |
+-----+-----------+------+---------+
| 2   | Khilan    | 25   | 1560    |
| 3   | kaushik   | 23   | 3000    |
| 3   | kaushik   | 23   | 1500    |
| 4   | Chaitali  | 25   | 2060    |
+-----+-----------+------+---------+
4 rows selected (2.64 seconds)


In [42]:
# Left outer join: every customer, with NULL where there is no order.
%%hive
SELECT c.ID, c.Name, o.Amount, o.`Date`
FROM Customers c
LEFT OUTER JOIN Orders o
ON c.ID = o.Customer_ID;


+-----+-----------+---------+----------------------+
| id  |   name    | amount  |         date         |
+-----+-----------+---------+----------------------+
| 1   | Ramesh    | NULL    | NULL                 |
| 2   | Khilan    | 1560    | 2009-11-20 00:00:00  |
| 3   | kaushik   | 3000    | 2009-10-08 00:00:00  |
| 3   | kaushik   | 1500    | 2009-10-08 00:00:00  |
| 4   | Chaitali  | 2060    | 2008-05-20 00:00:00  |
| 5   | Hardik    | NULL    | NULL                 |
| 6   | Komal     | NULL    | NULL                 |
| 7   | Muffy     | NULL    | NULL                 |
+-----+-----------+---------+----------------------+
8 rows selected (2.988 seconds)


In [43]:
# Right outer join: every order, with its customer.
%%hive
SELECT c.ID, c.Name, o.Amount, o.`Date`
FROM Customers c
RIGHT OUTER JOIN Orders o
ON c.ID = o.Customer_ID;


+-----+-----------+---------+----------------------+
| id  |   name    | amount  |         date         |
+-----+-----------+---------+----------------------+
| 3   | kaushik   | 3000    | 2009-10-08 00:00:00  |
| 3   | kaushik   | 1500    | 2009-10-08 00:00:00  |
| 2   | Khilan    | 1560    | 2009-11-20 00:00:00  |
| 4   | Chaitali  | 2060    | 2008-05-20 00:00:00  |
+-----+-----------+---------+----------------------+
4 rows selected (2.246 seconds)


In [44]:
# Full outer join: every row of both tables.
%%hive
SELECT c.ID, c.Name, o.Amount, o.`Date`
FROM Customers c
FULL OUTER JOIN Orders o
ON c.ID = o.Customer_ID;


+-----+-----------+---------+----------------------+
| id  |   name    | amount  |         date         |
+-----+-----------+---------+----------------------+
| 1   | Ramesh    | NULL    | NULL                 |
| 2   | Khilan    | 1560    | 2009-11-20 00:00:00  |
| 3   | kaushik   | 1500    | 2009-10-08 00:00:00  |
| 3   | kaushik   | 3000    | 2009-10-08 00:00:00  |
| 4   | Chaitali  | 2060    | 2008-05-20 00:00:00  |
| 5   | Hardik    | NULL    | NULL                 |
| 6   | Komal     | NULL    | NULL                 |
| 7   | Muffy     | NULL    | NULL                 |
+-----+-----------+---------+----------------------+
8 rows selected (3.9 seconds)


### 4.4 Analytic functions

*Sections 4.4 and 4.5 are self-study: they are not part of the tutorial recording.*

**Goal:** Compute values that depend on **other rows**, without collapsing them as `GROUP BY` does. An analytic (window) function is written `function() OVER (PARTITION BY … ORDER BY …)`:
- `PARTITION BY dept` restarts the calculation for each department (leave it out to use all rows);
- `ORDER BY salary DESC` sets the order inside each group.

| Function | Gives each row… |
| :-- | :-- |
| `RANK()` | its position; ties share a rank and the next rank is **skipped** (1, 1, 3) |
| `DENSE_RANK()` | its position; ties share a rank and **no** rank is skipped (1, 1, 2) |
| `ROW_NUMBER()` | a unique number 1, 2, 3 … even for ties |
| `CUME_DIST()` | the share of rows with a value **less than or equal** to this row's |
| `PERCENT_RANK()` | (rank − 1) / (rows − 1): 0 for the first row, 1 for the last |
| `NTILE(4)` | which of 4 roughly equal buckets the row falls in |

Hive can list its built-in functions for you (Hue shows the same list in its **Functions** panel). `SHOW FUNCTIONS LIKE` filters the list with a pattern, where `%` matches any text (as in `LIKE` in Section 4.5); `DESCRIBE FUNCTION` shows how to call one function.

**Expected:** `dense_rank`, `percent_rank` and `rank`; then `rank(x)`.

In [45]:
# List the built-in functions whose name contains "rank".
%%hive
SHOW FUNCTIONS LIKE '*rank*';


+-----------+
| tab_name  |
+-----------+
+-----------+
No rows selected (0.063 seconds)


In [46]:
# Explain what the rank function does.
%%hive
DESCRIBE FUNCTION rank;


+-----------+
| tab_name  |
+-----------+
| rank(x)   |
+-----------+
1 row selected (0.046 seconds)


In [47]:
# Rank all employees by salary, highest first; ties share a rank and the next rank is skipped.
%%hive
SELECT
    eid,
    name,
    salary,
    RANK() OVER (ORDER BY salary DESC) AS salary_rank
FROM employee;


+-------+--------------+----------+--------------+
|  eid  |     name     |  salary  | salary_rank  |
+-------+--------------+----------+--------------+
| 1202  | Manisha      | 45000.0  | 1            |
| 1201  | Gopal        | 45000.0  | 1            |
| 1204  | Kiran        | 40000.0  | 3            |
| 1203  | Masthanvali  | 40000.0  | 3            |
| 1205  | Kranthi      | 30000.0  | 5            |
+-------+--------------+----------+--------------+
5 rows selected (2.68 seconds)


**Expected:** Gopal and Manisha share rank 1, Masthanvali and Kiran share rank 3 (rank 2 is skipped), and Kranthi is 5.

In [48]:
# Rank salaries inside each department, without gaps.
%%hive
SELECT
    dept,
    salary,
    DENSE_RANK() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS dens_rank
FROM employee;


+----------+----------+------------+
|   dept   |  salary  | dens_rank  |
+----------+----------+------------+
| Admin    | 40000.0  | 1          |
| Admin    | 30000.0  | 2          |
| manager  | 45000.0  | 1          |
| reader   | 45000.0  | 1          |
| writer   | 40000.0  | 1          |
+----------+----------+------------+
5 rows selected (1.967 seconds)


In [49]:
# Number the rows inside each department.
%%hive
SELECT
    dept,
    salary,
    DENSE_RANK() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS dens_rank
FROM employee;


+----------+----------+------------+
|   dept   |  salary  | dens_rank  |
+----------+----------+------------+
| Admin    | 40000.0  | 1          |
| Admin    | 30000.0  | 2          |
| manager  | 45000.0  | 1          |
| reader   | 45000.0  | 1          |
| writer   | 40000.0  | 1          |
+----------+----------+------------+
5 rows selected (1.803 seconds)


In [50]:
# Share of employees earning the same or less than each employee.
%%hive
SELECT
    dept,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS row_no
FROM employee;


+----------+----------+---------+
|   dept   |  salary  | row_no  |
+----------+----------+---------+
| Admin    | 40000.0  | 1       |
| Admin    | 30000.0  | 2       |
| manager  | 45000.0  | 1       |
| reader   | 45000.0  | 1       |
| writer   | 40000.0  | 1       |
+----------+----------+---------+
5 rows selected (2.333 seconds)


**Expected:** 0.2 for 30000 (1 of 5 employees), 0.6 for 40000 (3 of 5) and 1.0 for 45000 (all 5).

In [51]:
# Rank and percent rank of each salary inside its department.
%%hive
SELECT
    dept,
    salary,
    CUME_DIST() OVER (
        ORDER BY salary
    ) AS cume_dist_value
FROM employee;


+----------+----------+------------------+
|   dept   |  salary  | cume_dist_value  |
+----------+----------+------------------+
| Admin    | 30000.0  | 0.2              |
| Admin    | 40000.0  | 0.6              |
| writer   | 40000.0  | 0.6              |
| reader   | 45000.0  | 1.0              |
| manager  | 45000.0  | 1.0              |
+----------+----------+------------------+
5 rows selected (1.887 seconds)


In [52]:
# Split each department's employees into 4 buckets by salary.
%%hive
SELECT
    dept,
    salary,
    RANK() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS dept_rank,
    PERCENT_RANK() OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS percent_rank
FROM employee;


+----------+----------+------------+---------------+
|   dept   |  salary  | dept_rank  | percent_rank  |
+----------+----------+------------+---------------+
| Admin    | 40000.0  | 1          | 0.0           |
| Admin    | 30000.0  | 2          | 1.0           |
| manager  | 45000.0  | 1          | 0.0           |
| reader   | 45000.0  | 1          | 0.0           |
| writer   | 40000.0  | 1          | 0.0           |
+----------+----------+------------+---------------+
5 rows selected (2.381 seconds)


**Aggregate functions** summarise many rows into one value:
- `count(*)` counts all rows; `count(salary)` counts the rows where `salary` is not `NULL`;
- `sum(salary)` adds the values; `sum(DISTINCT salary)` adds each different value only once.

**Expected:** `count(*)` 5, `count(salary)` 5, `sum(salary)` 200000.0, `sum(DISTINCT salary)` 115000.0 (45000 + 40000 + 30000).

In [53]:
# Count all rows, and the rows that have a salary.
%%hive
SELECT
    dept,
    salary,
    NTILE(4) OVER (
        PARTITION BY dept
        ORDER BY salary DESC
    ) AS bucket
FROM employee;


+----------+----------+---------+
|   dept   |  salary  | bucket  |
+----------+----------+---------+
| Admin    | 40000.0  | 1       |
| Admin    | 30000.0  | 2       |
| manager  | 45000.0  | 1       |
| reader   | 45000.0  | 1       |
| writer   | 40000.0  | 1       |
+----------+----------+---------+
5 rows selected (1.719 seconds)


In [54]:
# Add up all salaries, and each different salary once.
%%hive
SELECT
    COUNT(*) AS c0,
    COUNT(salary) AS c1
FROM employee;


+-----+-----+
| c0  | c1  |
+-----+-----+
| 5   | 5   |
+-----+-----+
1 row selected (1.677 seconds)


### 4.5 Additional exercise

**Goal:** Practise more functions on your own. Here are worked examples to start from; then write a few queries of your own on `Customers` and `Orders`.

| Example | What it shows |
| :-- | :-- |
| `count(DISTINCT …)` | How many different values a column has (4 departments, 4 designations) |
| `LIKE 'K%'` | `%` matches any text, so this finds names **starting** with K: Kiran and Kranthi |
| `LOWER(name) LIKE '%m%'` | `LIKE` is **case-sensitive**, so `name LIKE '%m%'` would find nothing (the M's are capitals). Lower-case the name first: Manisha and Masthanvali |
| `… LIKE '%m%' OR … NOT LIKE '%m%'` | Every row matches one side or the other, so all 5 rows come back |
| `RLIKE '.*(Gopal\|Kiran).*'` | A **regular expression**: names containing Gopal or Kiran |

In [55]:
# Count the different departments and designations.
%%hive
SELECT
    SUM(salary) AS c0,
    SUM(DISTINCT salary) AS c1
FROM employee;


+-----------+-----------+
|    c0     |    c1     |
+-----------+-----------+
| 200000.0  | 115000.0  |
+-----------+-----------+
1 row selected (3.178 seconds)


In [56]:
# Names starting with K.
%%hive
SELECT
    COUNT(DISTINCT dept),
    COUNT(DISTINCT designation)
FROM employee;


+------+------+
| _c0  | _c1  |
+------+------+
| 4    | 4    |
+------+------+
1 row selected (1.787 seconds)


In [57]:
# Names containing m in any case.
%%hive
SELECT *
FROM employee
WHERE name LIKE 'K%';


+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.33 seconds)


In [58]:
# Rows that contain m, or do not: together, every row.
%%hive
SELECT *
FROM employee
WHERE LOWER(name) LIKE '%m%';


+-------+--------------+----------+--------------+---------+
|  eid  |     name     |  salary  | designation  |  dept   |
+-------+--------------+----------+--------------+---------+
| 1202  | Manisha      | 45000.0  | Proof        | reader  |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer  |
+-------+--------------+----------+--------------+---------+
2 rows selected (0.217 seconds)


In [59]:
# Names matching a regular expression.
%%hive
SELECT *
FROM employee
WHERE LOWER(name) LIKE '%m%'
   OR LOWER(name) NOT LIKE '%m%';


+-------+--------------+----------+--------------+----------+
|  eid  |     name     |  salary  | designation  |   dept   |
+-------+--------------+----------+--------------+----------+
| 1201  | Gopal        | 45000.0  | Technical    | manager  |
| 1202  | Manisha      | 45000.0  | Proof        | reader   |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer   |
| 1204  | Kiran        | 40000.0  | Hr           | Admin    |
| 1205  | Kranthi      | 30000.0  | Op           | Admin    |
+-------+--------------+----------+--------------+----------+
5 rows selected (0.363 seconds)


In [60]:
# (Optional) Write your own query here, for example on the Customers or Orders table.
%%hive
SELECT *
FROM employee
WHERE name RLIKE '.*(Gopal|Kiran).*';


+-------+--------+----------+--------------+----------+
|  eid  |  name  |  salary  | designation  |   dept   |
+-------+--------+----------+--------------+----------+
| 1201  | Gopal  | 45000.0  | Technical    | manager  |
| 1204  | Kiran  | 40000.0  | Hr           | Admin    |
+-------+--------+----------+--------------+----------+
2 rows selected (0.396 seconds)


## 5. Optimise Hive queries

**Goal:** Make Hive read **less data**. Hive's main cost is reading files from HDFS, so the main optimisations arrange the files so that a query can skip the ones it does not need.

| Technique | How the files are arranged | Which queries benefit |
| :-- | :-- | :-- |
| **Columnar storage (ORC)** | Each file stores columns separately, with summary statistics | Queries that read a few columns or filter on a value |
| **Partitioning** | One HDFS **folder** per value of a column, e.g. one per department | Queries that filter on that column |
| **Bucketing** | A fixed number of **files** inside each folder, rows assigned by a hash of a column | Joins and samples on that column |

### 5.1 Indexing, and what replaced it

**Goal:** Understand why Hive no longer has indexes. Older versions of Hive (such as the one on the Cloudera VM) had `CREATE INDEX`; it was **removed in Hive 3.0**, so Hive 4 does not recognise it.

Indexes were replaced by **columnar file formats** such as **ORC**. An ORC file stores each column separately, and keeps the minimum and maximum value of each column for every block of rows. For `WHERE salary > 40000`, Hive reads those statistics and **skips** every block whose maximum salary is 40000 or less, which is what an index used to do, without having to build or rebuild anything.

`CREATE TABLE … STORED AS ORC AS SELECT …` (a "CTAS" statement) creates a new ORC table and fills it from a query in one step.

In [61]:
# Copy the employee table into a new table stored as ORC.
%%hive
CREATE TABLE employee_orc
STORED AS ORC
AS SELECT * FROM employee;


5 rows affected (2.293 seconds)


In [62]:
# Show how the new table is defined: note the ORC input and output formats.
%%hive
SHOW CREATE TABLE employee_orc;


+----------------------------------------------------+
|                   createtab_stmt                   |
+----------------------------------------------------+
| CREATE TABLE `employee_orc`(                       |
|   `eid` int,                                       |
|   `name` string,                                   |
|   `salary` float,                                  |
|   `designation` string,                            |
|   `dept` string)                                   |
| ROW FORMAT SERDE                                   |
|   'org.apache.hadoop.hive.ql.io.orc.OrcSerde'      |
| STORED AS INPUTFORMAT                              |
|   'org.apache.hadoop.hive.ql.io.orc.OrcInputFormat'  |
| OUTPUTFORMAT                                       |
|   'org.apache.hadoop.hive.ql.io.orc.OrcOutputFormat' |
| LOCATION                                           |
|   'hdfs://127.0.0.1:9000/user/hive/warehouse/userdb6.db/employee_orc' |
| TBLPROPERTIES (                         

Now run the same filter on both tables and compare the times beeline reports. Both return the same two employees, Gopal and Manisha (45000); the difference is how Hive finds them. The text table must read and check every line, while the ORC table first checks each block's minimum and maximum salary and reads only the blocks that can contain a salary above 40000. Here all five rows sit in one block (salaries 30000 to 45000), so that block is read. With five rows the times are almost equal, and the ORC query may even be slightly slower, because opening an ORC file has a small fixed cost; on millions of rows the skipped blocks make the difference.

In [63]:
# Filter the text table: Hive reads and checks every line.
%%hive
SELECT *
FROM employee
WHERE salary > 40000;


+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (0.274 seconds)


In [64]:
# Filter the ORC table: Hive can skip blocks using ORC's min/max statistics.
%%hive
SELECT *
FROM employee_orc
WHERE salary > 40000;


+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (0.432 seconds)


In [66]:
# List the ORC table's file: a binary file, not readable text like Sample.txt.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee_orc

Found 1 items
-rw-r--r--   1 root supergroup        801 2026-10-23 15:40 /user/hive/warehouse/userdb6.db/employee_orc/000000_0


### 5.2 Partitioning

**Goal:** Store each department in its own HDFS folder, so that a query on one department reads only that folder.

`PARTITIONED BY (dept STRING)` makes `dept` a **partition column**: it is not stored inside the data files, but in the folder name (`dept=Admin`).

In [67]:
# Create a table partitioned by department.
%%hive
CREATE TABLE employee_partitioned (
    eid INT,
    name STRING,
    salary FLOAT,
    designation STRING
)
PARTITIONED BY (dept STRING)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;


No rows affected (0.148 seconds)


The next statement fills the table with **dynamic partitioning**: Hive reads the `dept` value of each row and creates the matching partition itself. By default Hive runs in **strict mode**, which requires at least one partition value to be written out in the statement. `SET` switches this session to **non-strict mode**, which allows every partition to be created dynamically. The setting lasts for the rest of the session.

In `INSERT OVERWRITE TABLE … PARTITION (dept) SELECT …`, the partition column must come **last** in the `SELECT` list.

In [68]:
# Allow every partition to be created dynamically (for this session).
%%hive
SET hive.exec.dynamic.partition.mode=nonstrict;


No rows affected (0.02 seconds)


In [69]:
# Fill the partitioned table from employee; Hive creates one partition per department.
%%hive
INSERT OVERWRITE TABLE employee_partitioned
PARTITION (dept)
SELECT eid, name, salary, designation, dept
FROM employee;


5 rows affected (4.705 seconds)


In [70]:
# List the partitions Hive created.
%%hive
SHOW PARTITIONS employee_partitioned;


+---------------+
|   partition   |
+---------------+
| dept=Admin    |
| dept=manager  |
| dept=reader   |
| dept=writer   |
+---------------+
4 rows selected (0.128 seconds)


In [71]:
# List the table's HDFS folder: one sub-folder per department.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee_partitioned

Found 4 items
drwxr-xr-x   - root supergroup          0 2026-10-23 15:43 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=Admin
drwxr-xr-x   - root supergroup          0 2026-10-23 15:43 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=manager
drwxr-xr-x   - root supergroup          0 2026-10-23 15:43 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=reader
drwxr-xr-x   - root supergroup          0 2026-10-23 15:43 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=writer


In [72]:
# Query one partition: Hive reads only the dept=Admin folder.
%%hive
SELECT *
FROM employee_partitioned
WHERE dept = 'Admin';


+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.581 seconds)


**Expected:** Kiran and Kranthi.

**See the difference.** `EXPLAIN DEPENDENCY` lists the data a query will read, without running it. Compare the two: on the partitioned table the input is only the `dept=Admin` partition, while on `employee` it is the whole table. This skipping is called **partition pruning**.

In [73]:
# Which data does the query read on the partitioned table?
%%hive
EXPLAIN DEPENDENCY
SELECT *
FROM employee_partitioned
WHERE dept = 'Admin';


+----------------------------------------------------+
|                      Explain                       |
+----------------------------------------------------+
| {"input_tables":[{"tablename":"userdb6@employee_partitioned","tabletype":"MANAGED_TABLE"}],"input_partitions":[{"partitionName":"userdb6@employee_partitioned@dept=Admin"}]} |
+----------------------------------------------------+
1 row selected (0.37 seconds)


In [75]:
# Which data does the same query read on the unpartitioned table?
%%hive
EXPLAIN DEPENDENCY
SELECT *
FROM employee
WHERE dept = 'Admin';


+----------------------------------------------------+
|                      Explain                       |
+----------------------------------------------------+
| {"input_tables":[{"tablename":"userdb6@employee","tabletype":"MANAGED_TABLE"}],"input_partitions":[]} |
+----------------------------------------------------+
1 row selected (0.274 seconds)


### 5.3 Bucketing

**Goal:** Split each partition into a fixed number of files. `CLUSTERED BY (eid) INTO 4 BUCKETS` sends every row to one of 4 files, chosen by a hash of its `eid`. Rows with the same `eid` always land in the same bucket, so joins on `eid` can match bucket to bucket, and `TABLESAMPLE` can read a single bucket.

In [76]:
# Create a table partitioned by department and bucketed by eid into 4 files.
%%hive
CREATE TABLE employee_bucketed (
    eid INT,
    name STRING,
    salary FLOAT,
    designation STRING
)
PARTITIONED BY (dept STRING)
CLUSTERED BY (eid) INTO 4 BUCKETS
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;


No rows affected (0.251 seconds)


In [77]:
# Fill the bucketed table from the partitioned table.
%%hive
INSERT OVERWRITE TABLE employee_bucketed
PARTITION (dept)
SELECT eid, name, salary, designation, dept
FROM employee_partitioned;


5 rows affected (4.924 seconds)


In [78]:
# Query the bucketed table.
%%hive
SELECT *
FROM employee_bucketed
WHERE eid < 1205;


+-------+--------------+----------+--------------+----------+
|  eid  |     name     |  salary  | designation  |   dept   |
+-------+--------------+----------+--------------+----------+
| 1204  | Kiran        | 40000.0  | Hr           | Admin    |
| 1201  | Gopal        | 45000.0  | Technical    | manager  |
| 1202  | Manisha      | 45000.0  | Proof        | reader   |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer   |
+-------+--------------+----------+--------------+----------+
4 rows selected (0.626 seconds)


**Expected:** Gopal, Manisha, Masthanvali and Kiran. Then look inside one partition folder: its rows are spread over bucket files named `000000_0`, `000001_0` …

In [79]:
# List the bucket files inside the dept=Admin partition.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin

Found 4 items
-rw-r--r--   1 root supergroup          0 2026-10-23 15:45 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000000_0
-rw-r--r--   1 root supergroup         22 2026-10-23 15:45 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000001_0
-rw-r--r--   1 root supergroup         24 2026-10-23 15:45 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000002_0
-rw-r--r--   1 root supergroup          0 2026-10-23 15:45 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000003_0


Bucketing helps most with **joins** on the bucketing column. Because rows with the same `eid` always land in the same bucket, Hive can join a large bucketed table one bucket at a time instead of comparing every row with every other row. Try a join on `eid`, and compare its time with the joins in Section 4.3.

**Expected:** all five employees, each with their salary.

In [80]:
# Join the bucketed table with the partitioned table on the bucketing column eid.
%%hive
SELECT b.eid, b.name, b.salary
FROM employee_bucketed b
JOIN employee_partitioned p
ON b.eid = p.eid;


+-------+--------------+----------+
|  eid  |     name     |  salary  |
+-------+--------------+----------+
| 1205  | Kranthi      | 30000.0  |
| 1204  | Kiran        | 40000.0  |
| 1201  | Gopal        | 45000.0  |
| 1202  | Manisha      | 45000.0  |
| 1203  | Masthanvali  | 40000.0  |
+-------+--------------+----------+
5 rows selected (3.207 seconds)


### 5.4 Combine partitioning and bucketing

**Goal:** Partition by department and bucket by salary, then compare the tables you have built.

In [81]:
# Create a table partitioned by department and bucketed by salary into 5 files.
%%hive
CREATE TABLE employee_optimized (
    eid INT,
    name STRING,
    salary FLOAT,
    designation STRING
)
PARTITIONED BY (dept STRING)
CLUSTERED BY (salary) INTO 5 BUCKETS
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE;


No rows affected (0.652 seconds)


In [82]:
# Fill the optimised table from employee.
%%hive
INSERT OVERWRITE TABLE employee_optimized
PARTITION (dept)
SELECT eid, name, salary, designation, dept
FROM employee;


5 rows affected (4.715 seconds)


In [83]:
# Query the optimised table.
%%hive
SELECT *
FROM employee_optimized
WHERE dept = 'Admin';


+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.416 seconds)


**Compare query performance.** Run the same query on `employee`, `employee_partitioned`, `employee_bucketed` and `employee_optimized`, and compare the time beeline reports (`… rows selected (x seconds)`) and the `EXPLAIN DEPENDENCY` inputs. Then write one paragraph describing the differences.

With only five rows, the times are dominated by Hive's fixed start-up costs, so they will be close. Explain what you would expect with millions of rows, and why: which tables let Hive skip folders or files, and for which queries?

In [84]:
# (optional) Run the same query on the four tables and compare the reported times.
%%hive
SELECT * FROM employee
WHERE dept = 'Admin';

+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.199 seconds)


## 6. Extra exercise: Apache Pig and the Grunt shell

**Goal:** Count how often each word appears in `wordcount_simple.txt`, using **Apache Pig**.

Pig is another high-level tool for Hadoop. Instead of SQL, you write **Pig Latin**: a sequence of steps, each giving a name (a **relation**) to the result of transforming the previous one. Pig turns the steps into MapReduce jobs. Its interactive shell is called **Grunt** (`grunt>`).

`pig -x local` runs Pig in **local mode**: it reads files from the local disk instead of HDFS, which is ideal for trying out a script. Each `%%pig` cell types into the same Grunt shell, so a relation created in one cell can be used in the next.

**Expected output:** `Pig is ready ✓`. The first run downloads Pig (about 470 MB).

In each step below, the cell runs one Pig Latin statement and then looks at the result with `describe` and `dump`, so you can see exactly what the statement changed.

In [85]:
# @title ▶ 6.1 Install Apache Pig 0.18.0 and turn on %%pig cells
# Pig 0.18 is the first release that runs on Hadoop 3.
PIG_VERSION = '0.18.0'
PIG = LAB / f'pig-{PIG_VERSION}'
PIG_WORK = LAB / 'pig-work'
PIG_WORK.mkdir(exist_ok=True)

# Download, verify and unpack Pig only once per current VM.
if not (PIG / '.verified').exists():
    print(f'Downloading Apache Pig {PIG_VERSION} (about 470 MB) …')
    archive = LAB / 'pig.tar.gz'
    download_apache(f'pig/pig-{PIG_VERSION}/pig-{PIG_VERSION}.tar.gz', archive,
                    ('.sha512', '.sha256', '.sha1', '.md5'), required=False)
    print('Installing Pig …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    for script in (PIG / 'bin').iterdir():
        script.chmod(script.stat().st_mode | 0o111)
    (PIG / '.verified').write_text('ok')
    archive.unlink()

# Pig needs Apache Commons Collections 3, which Hadoop 3.5 no longer ships; add it from Maven Central.
COLLECTIONS = PIG / 'lib/commons-collections-3.2.2.jar'
if not COLLECTIONS.exists():
    download_verified('https://repo1.maven.org/maven2/commons-collections/commons-collections/3.2.2/'
                      'commons-collections-3.2.2.jar', COLLECTIONS, ('.sha1',))

# Show only Pig's warnings and errors, so each statement prints just its result.
PIG_LOG4J = PIG_WORK / 'log4j.properties'
PIG_LOG4J.write_text('log4j.rootLogger=WARN, A\n'
                     'log4j.appender.A=org.apache.log4j.ConsoleAppender\n'
                     'log4j.appender.A.target=System.err\n'
                     'log4j.appender.A.layout=org.apache.log4j.PatternLayout\n'
                     'log4j.appender.A.layout.ConversionPattern=%p %c{1}: %m%n\n'
                     'log4j.logger.org.apache.hadoop=ERROR\n'
                     'log4j.logger.org.apache.pig.data.SchemaTupleBackend=ERROR\n')

# Environment for Pig: Java 17 settings, 1 GB of memory, and a plain terminal.
PIG_ENV = {**os.environ, 'PIG_HOME': str(PIG), 'PIG_OPTS': OPENS, 'PIG_HEAPSIZE': '1024', 'TERM': 'dumb',
           'PIG_CLASSPATH': str(COLLECTIONS)}
os.environ['PATH'] = f'{PIG}/bin:' + os.environ['PATH']

# Close a Grunt shell left over from an earlier run of this cell.
if 'PIG_SESSION' in globals() and PIG_SESSION.child is not None and PIG_SESSION.child.isalive():
    PIG_SESSION.child.terminate(force=True)

# One Grunt shell in local mode (-x local reads files from Colab's own disk), kept open for the notebook.
PIG_SESSION = ShellSession(
    [str(PIG / 'bin/pig'), '-x', 'local', '-4', str(PIG_LOG4J)],
    prompt=r'grunt> ', run_command='run {}', echo_command='sh echo {}', env=PIG_ENV, cwd=PIG_WORK)

# Remove full-line -- comments from a cell of Pig Latin.
def pig_latin(cell):
    return '\n'.join(l for l in cell.strip().split('\n') if not l.strip().startswith(('--', '#')))

# Notebook shortcut: a %%pig cell types its Pig Latin into the Grunt shell.
@register_cell_magic
def pig(line, cell):
    text = pig_latin(cell)
    if not text.strip():
        print('(Nothing to run yet: type a Pig Latin statement in this cell, ending with ;)')
        return
    # quit is typed straight into Grunt, as in a terminal, and closes the shell.
    if text.strip().rstrip(';').strip().lower() == 'quit':
        if PIG_SESSION.child is not None and PIG_SESSION.child.isalive():
            PIG_SESSION.child.sendline('quit')
            PIG_SESSION.child.expect(pexpect.EOF, timeout=60)
            PIG_SESSION.child.close()
        PIG_SESSION.child = None
        print('Grunt shell closed. The next %%pig cell will start a new one.')
        return
    # Show results and errors; hide Pig's warnings.
    output = PIG_SESSION.run(text)
    print('\n'.join(l for l in output.split('\n') if not re.match(r'(WARN|INFO) ', l)).strip('\n'))

# Start the Grunt shell now, so the first %%pig cell answers quickly.
if PIG_SESSION.child is None or not PIG_SESSION.child.isalive():
    PIG_SESSION.start()
print('Pig is ready ✓  Grunt shell in local mode · %%pig cells are ready ✓')

  Trying https://dlcdn.apache.org …
  Problem: The read operation timed out. Retrying …
  Trying https://dlcdn.apache.org …
  Problem: The read operation timed out. Retrying …
  Trying https://downloads.apache.org …
  Download: 0%  (1 of 468 MB)
  Download: 10%  (47 of 468 MB)
  Download: 20%  (94 of 468 MB)
  Download: 30%  (141 of 468 MB)
  Download: 40%  (188 of 468 MB)
  Download: 50%  (235 of 468 MB)
  Download: 60%  (282 of 468 MB)
  Download: 70%  (329 of 468 MB)
  Download: 80%  (376 of 468 MB)
  Download: 90%  (423 of 468 MB)
  Download: 100%  (468 of 468 MB)
  MD5 checksum verified: pig.tar.gz
Installing Pig …
  SHA1 checksum verified: commons-collections-3.2.2.jar
Pig is ready ✓  Grunt shell in local mode · %%pig cells are ready ✓


### 6.2 Load the file

**Goal:** Read the text file into a relation called `Pigdata`, with one field per line: `line`, of type `chararray` (text).

Two commands let you look at a relation after each step:

| Command | Shows | Runs a job? |
| :-- | :-- | :-- |
| `describe Pigdata;` | The relation's **schema**: its fields and their types | No, it answers at once |
| `dump Pigdata;` | The relation's **data**, one tuple `( … )` per line | Yes: Pig only records statements until you ask for a result, and `dump` makes it run them |

**Expected:** `Pigdata: {line: chararray}`, then the file's single line as one tuple: `(Class of CN7031; Apache Pig is a high-level platform … Apache Spark.)`.

In [86]:
# Load the text file, then show the new relation's schema and data.
%%pig
Pigdata = LOAD '/content/wordcount_simple.txt'
          AS (line:chararray);


>> grunt> 


### 6.3 Split the lines into words

**Goal:** `TOKENIZE(line, ' ')` splits each line at spaces into a **bag** (a collection) of words, and `FLATTEN` turns the bag into one record per word.

**What changed:** one record holding a whole line became **39 records**, one per word, in the order they appear in the text. The field is now called `word`.

**Expected:** `Pigwords: {word: chararray}`, then `(Class)`, `(of)`, `(CN7031;)`, `(Apache)`, `(Pig)` … one word per line.

In [87]:
# Split every line at spaces into one record per word, then show the result.
%%pig
DESCRIBE Pigdata;


Pigdata: {line: chararray}


### 6.4 Group the words

**Goal:** `GROUP … BY word` collects all copies of the same word into one group.

**What changed:** the 39 word records became **31 groups**, one per different word. Each group is a tuple with two fields: the key, always named `group`, and a bag holding every record of that word. The schema shows the bag nested inside the tuple.

**Expected:** `Piggrouping: {group: chararray,Pigwords: {(word: chararray)}}`, then lines such as `(Pig,{(Pig),(Pig),(Pig)})` and `(a,{(a)})`. Pig does not promise any particular order of groups.

In [89]:
%%pig
# Collect identical words into one group each, then show the groups and their bags.
Piggrouping = GROUP Pigwords BY word;
DESCRIBE Piggrouping;
DUMP Piggrouping;

ERROR Grunt: ERROR 1200: Pig script failed to parse: <line 3, column 20> Undefined alias: Pigwords
Details at logfile: /content/cn7031-week4-hive/pig-work/pig_1792770650578.log


### 6.5 Count each group

**Goal:** For each group, output the word (`group`) and the number of records in its bag (`COUNT(Pigwords)`).

**What changed:** each bag is replaced by a single number. The count has no name, so the schema lists only its type, `long`.

**Expected:** `Pig_word_count: {group: chararray,long}`. `describe` does not run a job: the counting happens in the next step.

In [90]:
# Replace each group's bag with the number of words in it, then show the new schema.
%%pig
DUMP Pigdata;


(Class of CN7031; Apache Pig is a high-level platform for creating programs that run on Apache Hadoop. The language for this platform is called Pig Latin. Pig can execute its Hadoop jobs in MapReduce, Apache Tez, or Apache Spark.)


### 6.6 Show the result

**Goal:** `dump` runs all the steps as a job and prints the result. Each line is a **tuple** `(word,count)`.

**Expected:** 31 pairs, including `(Pig,3)`, `(Apache,4)`, `(is,2)` and `(platform,2)`. Punctuation stays attached to words (`Hadoop.` and `Hadoop` are counted separately), because the text was split only at spaces.

In [91]:
# Run the steps and print every (word,count) pair.
%%pig
Pigwords = FOREACH Pigdata
           GENERATE FLATTEN(TOKENIZE(line, ' ')) AS word;


>> grunt> 



### 6.7 Leave the Grunt shell

**Goal:** Close Pig and look at the folder it worked in. `quit` ends the Grunt shell, as `Ctrl + C` does in a terminal. The next `%%pig` cell would start a new Grunt shell, without the relations you created.

The folder holds `cell_1.txt`, `cell_2.txt` … (the Pig Latin from each of your `%%pig` cells, which the notebook passed to Grunt) and Pig's logging settings. When a statement **fails**, Pig also writes a log file here named `pig_<number>.log`, with the full details of the error; after an error-free run there is none.

In [92]:
# Leave the Grunt shell.
%%pig
DESCRIBE Pigwords;


Pigwords: {word: chararray}


In [93]:
# List the files in Pig's working folder: your cell scripts, and a pig_….log file for any error.
%%pig
DUMP Pigwords;

(Class)
(of)
(CN7031;)
(Apache)
(Pig)
(is)
(a)
(high-level)
(platform)
(for)
(creating)
(programs)
(that)
(run)
(on)
(Apache)
(Hadoop.)
(The)
(language)
(for)
(this)
(platform)
(is)
(called)
(Pig)
(Latin.)
(Pig)
(can)
(execute)
(its)
(Hadoop)
(jobs)
(in)
(MapReduce,)
(Apache)
(Tez,)
(or)
(Apache)
(Spark.)


## 7. Clean up (optional)

**Goal:** Remove what this lab created. `DROP DATABASE … CASCADE` deletes the database together with all its tables, and their folders in HDFS. Colab deletes everything when the runtime ends, so this step is optional.

In [94]:
# Delete the userdb6 database with all its tables.
%%hive
DROP DATABASE userdb6 CASCADE;


No rows affected (3.153 seconds)


## Check your understanding

1. `ALTER TABLE … ADD COLUMNS` filled the new `dept` column for rows that already existed. Why could Hive do this without rewriting the data file?
2. What is stored in the **metastore**, and what is stored in **HDFS**? What would be lost if the metastore were deleted?
3. Why did `Sample.txt` need its line endings fixed before `WHERE dept = 'Admin'` could work?
4. Why do some queries return in a fraction of a second while others take several seconds?
5. Explain the difference between `RANK`, `DENSE_RANK` and `ROW_NUMBER` using the salaries 45000, 45000 and 40000.
6. A table holds ten years of sales and most queries ask about a single month. Would you partition or bucket it, and by which column? What could go wrong if you partitioned by customer id instead?